<a href="https://colab.research.google.com/github/ekosup/pkn-stan-padaa/blob/main/exercise/w5/data_audit_latihan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Audit Data dengan SQL
## Latihan Mandiri — Google Colab + NeonDB

**Fokus: SQL.** Python cuma alat bantu — kamu menulis query, `q()` mengeksekusi.

> Dataset: ~8.000 transaksi General Ledger + ~600 pembayaran AP buatan
> (sudah tersedia ke NeonDB).
> ⏱️ Estimasi: 150 menit (6 modul × 25 menit)

### Cara pakai
1. Isi `NEON_CONN_AUDIT` di sel SETUP dengan connection string dari dosen
2. Jalankan sel SETUP
3. Baca penjelasan tiap modul, tulis query SQL di sel jawaban
4. Jalankan sel untuk melihat hasil

---
## ⚙️ SETUP — Jalankan sekali

In [ ]:
# !pip install psycopg2-binary -q

In [5]:
import psycopg2
import time
import pandas as pd
from google.colab import userdata


# ⚠️ GANTI dengan connection string dari dosen
NEON_CONN_AUDIT = userdata.get("NEON_CONN_AUDIT")

conn = psycopg2.connect(NEON_CONN_AUDIT)

class Database:
    def __init__(self, conn_string):
        self.conn = psycopg2.connect(conn_string)

    def close(self):
        self.conn.close()

    def run(self, sql, params=None):
        start = time.perf_counter()

        with self.conn.cursor() as cur:
            try:
                cur.execute(sql, params)

                elapsed = time.perf_counter() - start

                if cur.description:
                    cols = [c.name for c in cur.description]
                    rows = cur.fetchall()

                    print(f"✅ {len(rows)} row(s) ({elapsed:.3f}s)")
                    return pd.DataFrame(rows, columns=cols)

                self.conn.commit()
                print(f"✅ {cur.rowcount} row(s) affected ({elapsed:.3f}s)")

            except Exception as e:
                self.conn.rollback()
                print(f"❌ SQL Error:\n{e}")
                raise

    def tables(self):
        return self.run("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema='public'
            ORDER BY table_name
        """)

    def describe(self, table):
        return self.run("""
            SELECT
                column_name,
                data_type,
                is_nullable
            FROM information_schema.columns
            WHERE table_name=%s
            ORDER BY ordinal_position
        """, (table,))

    def schema(self):
        return self.run("""
            SELECT
                table_name,
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_schema='public'
            ORDER BY table_name, ordinal_position
        """)

    def explain(self, sql):
        return self.run(f"EXPLAIN ANALYZE {sql}")


db = Database(NEON_CONN_AUDIT)

def q(sql):
    return db.run(sql)

def schema():
    return db.schema()

def tables():
    return db.tables()

def desc(table):
    return db.describe(table)

print("✅ Database siap!")

✅ Database siap!


In [7]:
# 🔍 Intip isi tabel — biar kenal bentuk datanya
print('📋 Chart of Accounts (akun RESMI):')
display(q("SELECT * FROM chart_of_accounts ORDER BY account_code"))

print('📋 GL Lines (5 sampel):')
display(q("SELECT line_id, entry_date, account_code, debit, invoice_no, created_by FROM gl_lines ORDER BY line_id LIMIT 5"))

📋 Chart of Accounts (akun RESMI):
✅ 10 row(s) (0.145s)


,account_code,account_name
0,1000,Kas
1,1100,Bank
2,1200,Piutang Usaha
3,1300,Persediaan
4,2000,Utang Usaha
5,4000,Pendapatan
6,5000,Beban Pokok
7,6000,Beban Operasional
8,6100,Beban Perjalanan
9,6200,Beban Konsultan


📋 GL Lines (5 sampel):
✅ 5 row(s) (0.082s)


,line_id,entry_date,account_code,debit,invoice_no,created_by
0,1,2024-07-04 12:15:00,6200,2187053.63,INV00001,eka
1,2,2024-03-05 15:35:00,6100,10607291.61,INV00002,SYSTEM
2,3,2024-12-19 12:25:00,4000,12963517.23,inv00003,andi
3,4,2024-06-22 03:14:00,1000,15796018.78,INV00004,andi
4,5,2024-06-18 11:17:00,1300,18097646.52,INV00005,budi


---
## MODUL 2.1 — Extract & Profiling

**Konsep:** Chart of Accounts = daftar RESMI akun. Gl Lines = transaksi aktual.
Tugas: kenali bentuk data, lalu cari transaksi yang pakai akun TIDAK RESMI.

**Teknik SQL:** `SELECT`, `DISTINCT`, `LEFT JOIN ... WHERE ... IS NULL`

### 📝 Soal 1: Akun apa saja yang pernah dipakai?
Tampilkan SEMUA kode akun UNIK yang muncul di `gl_lines`. Gunakan `DISTINCT`.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 2: Cari Akun HANTU
Temukan transaksi di `gl_lines` yang kode akunnya **tidak terdaftar** di
`chart_of_accounts`. Gunakan `LEFT JOIN` — akun hantu akan muncul dengan `NULL`
di sisi kanan.

Tampilkan: `line_id`, `account_code`, `debit`, `entry_date`.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 3: Berapa BANYAK akun hantu?
Hitung jumlah total transaksi yang memakai akun hantu.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 💡 Soal 4 (Bonus): Profiling Vendor
Siapa 5 vendor dengan transaksi terbanyak di `gl_lines`?
Tampilkan `vendor_id` dan jumlah transaksi, urutkan dari terbesar.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

---
## MODUL 2.2 — Data Cleansing

**Konsep:** Komputer membaca `"  INV00042"` dan `"inv00042"` sebagai DUA hal
berbeda, walau manusia tahu itu sama. Sebelum analisis lanjutan, data HARUS
diseragamkan.

**Teknik:** `TRIM`, `UPPER`, `COALESCE`, `CAST`, `REGEXP_REPLACE`, `CASE WHEN`

### 📝 Soal 1: Identifikasi Data Kotor
Tampilkan `invoice_no` yang memiliki spasi di depan/belakang atau huruf kecil.
Tambahkan kolom `terlihat` dengan format `'[' || invoice_no || ']'` supaya
spasi kelihatan.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 2: Bersihkan dengan TRIM + UPPER
Tampilkan invoice SEBELUM dan SESUDAH dibersihkan.
Kolom: `line_id`, `sebelum` (pakai bracket), `sesudah` (pakai bracket).

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 3: COALESCE untuk kolom kosong
Buat kolom `pembuat_bersih`: jika `created_by` NULL atau string kosong → isi `'TIDAK DIKETAHUI'`.
Tampilkan `line_id`, `created_by` asli, dan `pembuat_bersih`. 15 baris pertama.

> `COALESCE(NULLIF(TRIM(created_by), ''), 'TIDAK DIKETAHUI')`

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 4: Ekstrak Angka + Kategori Nilai
Dari `invoice_no` yang sudah dibersihkan, ekstrak ANGKA-nya sebagai integer.
Lalu beri label: `'BESAR'` (≥ 50 juta), `'SEDANG'` (≥ 5 juta), `'KECIL'` (sisanya).

> Ekstrak angka: `CAST(REGEXP_REPLACE(UPPER(TRIM(invoice_no)), '[^0-9]', '', 'g') AS INTEGER)`

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

---
## MODUL 2.3 — Deteksi Anomali Transaksi

**Konsep:** Filter transaksi mencurigakan berdasarkan waktu dan nilai.
Belum "vonis", baru mempersempit kandidat investigasi.

**Teknik:** `EXTRACT(HOUR FROM ...)`, filter nilai bulat, `BETWEEN`, `IN`

### 📝 Soal 1: Transaksi di Luar Jam Kerja
Cari transaksi yang terjadi jam 23:00–04:00. Tampilkan `line_id`, `entry_date`,
`EXTRACT(HOUR FROM entry_date)` AS `jam`, `account_code`, `debit`, `created_by`.
Urutkan berdasarkan waktu.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 2: Siapa user dengan transaksi after-hours terbanyak?
Group by `created_by`, hitung jumlah after-hours-nya.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 3: Nilai Bulat Besar Mencurigakan
Cari transaksi dengan `debit` ≥ 100 juta yang nilainya BULAT.
Urutkan dari terbesar.

> Bulat = `debit = ROUND(debit, -6)` atau `MOD(debit::BIGINT, 1000000) = 0`

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 4: BETWEEN + IN
Cari transaksi di **Maret 2024** dengan kode akun beban perjalanan/konsultan/sewa
(`6100`, `6200`, `6300`). Tampilkan `line_id`, `entry_date`, `account_code`, `debit`.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

---
## MODUL 3.1 — Agregasi & Batas Materialitas

**Konsep:** Kelompokkan transaksi per vendor dulu (`GROUP BY`), baru hitung
totalnya. `HAVING` = alarm setelah agregasi — "total ke satu vendor
melebihi batas tender".

**Teknik:** `GROUP BY`, `SUM`, `COUNT`, `MIN`, `MAX`, `HAVING`

### 📝 Soal 1: Total Pembayaran per Vendor
Dari `ap_payments`, hitung jumlah transaksi dan total bayar per vendor.
Urutkan dari total terbesar. Tampilkan 10 teratas.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 2: Alarm Materialitas
Asumsikan batas tender = Rp250 juta. Vendor mana saja yang TOTAL bayarnya
melebihi batas ini? Urutkan dari yang terbesar.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 3: Deteksi Pola SPLIT
Pola split: vendor memecah transaksi jadi kecil-kecil (< 100jt per transaksi)
supaya lolos radar, tapi totalnya justru jauh di atas ambang.

Cari vendor yang memenuhi: **semua transaksi < 100jt** tapi **total > 300jt**.
Tampilkan `vendor_id`, jumlah transaksi, nilai min, nilai max, total.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

---
## MODUL 3.2 — Subquery untuk Investigasi

**Konsep:** Subquery = kumpulkan "daftar tersangka" dulu, baru pakai daftar itu
untuk menyaring data utama.

**Teknik:** subquery `IN`, subquery `EXISTS`, correlated subquery, `JOIN` multi-tabel

### 📝 Soal 1: Subquery dengan IN
Cari transaksi di `gl_lines` yang vendornya termasuk dalam daftar vendor dengan
total pembayaran AP > 250jt. Gunakan subquery `IN`.

Tampilkan: `line_id`, `vendor_id`, `debit`, `account_code`. Urutkan debit terbesar.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 2: Correlated Subquery
Cari transaksi yang debitnya > 3× rata-rata debit dari **akun yang sama**.

Tampilkan: `line_id`, `account_code`, `debit`. Urutkan debit terbesar.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 3: Siphoning Karyawan-Vendor
**Naratif:** Ada kecurigaan karyawan yang merangkap sebagai vendor fiktif.
Buktinya? Rekening bank atau alamat yang SAMA.

Cari karyawan yang `bank_account` atau `address`-nya cocok dengan vendor.
Tampilkan: `employee_id`, `employee_name`, `vendor_id`, `vendor_name`, dan
kolom yang cocok (pakai `CASE WHEN`).

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 💡 Soal 4 (Bonus): EXISTS vs IN
Tulis ulang Soal 1 menggunakan `EXISTS` (bukan `IN`). Hasil harus identik.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

---
## MODUL 3.3 — Analisis Duplikasi & Gap

**Konsep:** Self-join = membandingkan tabel dengan DIRINYA SENDIRI.
Window function = menghitung nilai "tetangga" dalam urutan.

**Teknik:** self-join, window function `LAG`, `UNION ALL`

### 📝 Soal 1: Deteksi Double Payment (Self-Join)
Cari pasangan pembayaran ganda di `ap_payments` dengan kriteria:
- **vendor_id** sama, **pay_date** sama, **amount** sama
- **invoice_no** BEDA

Tampilkan: `payment_id_1`, `payment_id_2`, `vendor_id`, `pay_date`, `amount`,
`invoice_1`, `invoice_2`. Pastikan tiap pasangan hanya muncul SEKALI
(syarat: `a.payment_id < b.payment_id`).

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 2: Gap Analysis Nomor Invoice
Setelah invoice dibersihkan, cari LOMPATAN nomor urut pakai window function.

Gunakan CTE (`WITH bersih AS (...)`) lalu `LAG(no_urut) OVER (ORDER BY no_urut)`.
Tampilkan nomor setelah gap dan besar lompatannya.

> Ekstrak angka: `CAST(REGEXP_REPLACE(UPPER(TRIM(invoice_no)), '[^0-9]', '', 'g') AS INTEGER)`

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 📝 Soal 3: Rekap Semua Temuan
Gabungkan semua hasil investigasi jadi satu laporan ringkas:
- Jumlah double payment
- Jumlah akun hantu
- Jumlah transaksi after-hours

Gunakan `UNION ALL` dengan label `jenis_temuan`.

In [ ]:
q("""
    -- ✏️ TULIS QUERY KAMU DI SINI

""")

### 🧠 Soal 4: Refleksi
Dari semua teknik yang sudah dipelajari (Modul 2.1–3.3), menurutmu teknik mana
yang PALING powerful untuk auditor? Jelaskan dalam 2-3 kalimat kenapa.

(Jawaban bebas — tulis di sel bawah ini.)

In [ ]:
# ✏️ TULIS JAWABAN REFLEKSI KAMU DI SINI
# (Tidak perlu SQL, tulis sebagai komentar Python seperti ini)



---
## ✅ Selesai!

Kamu sudah menyelesaikan 6 modul latihan audit data dengan SQL. Ringkasan
teknik yang sudah dipelajari:

| Modul | Teknik | Untuk Mendeteksi |
|-------|--------|-----------------|
| 2.1 | SELECT, DISTINCT, LEFT JOIN | Akun hantu |
| 2.2 | TRIM, UPPER, COALESCE, CAST | Data kotor |
| 2.3 | EXTRACT, BETWEEN, IN | Transaksi mencurigakan |
| 3.1 | GROUP BY, HAVING | Materialitas & split |
| 3.2 | Subquery, EXISTS, JOIN | Siphoning karyawan |
| 3.3 | Self-join, LAG, UNION ALL | Double payment & gap |

> 💡 **Pesan penutup:** SQL hanyalah ALAT PENYARING. Ia mempersempit "siapa
> yang perlu diperiksa lebih lanjut" — vonis tetap di tangan auditor manusia.
> Tidak ada query yang otomatis membuktikan fraud tanpa investigasi lanjutan.